In [ ]:
# pip install 

#!pip install argon2-cffi

# dont know what the assignment is looking for 
#actually may use os for this to use shell commands

In [ ]:
# authentication method 

import json
import time
import os 
import secrets
import hmac
import hashlib
from argon2 import PasswordHasher
from argon2.exceptions import VerifyMismatchError

# argon2 hasher 
ph = PasswordHasher()
DB_FILE = 'users_database.json'

#loading user database
def load_user_data():
    if not os.path.exists(DB_FILE):
        return {} #empty dictionary 
    with open(DB_FILE, 'r') as file:
        return json.load(file)
    
# take user database and dump in JSON file    
def save_user_data(data):
    with open(DB_FILE, 'w') as file:
        json.dump(data, file, indent=4)


# save user data in database
def register_user(username, password):
    users = load_user_data()

    #check if user is already registered
    if username in users:
        print("Error: User already exists.")
        return False
    
    #check if password meets requirements 
    if len(password) < 16 or password.lower() == username.lower():
        print("Error: password must be 16+ chars and not your username.")
        return False
    
    #hash the password
    pw_hash = ph.hash(password)
    
    #Data chunk for new user to be saved to db 
    users[username] = {
        "password": pw_hash,
        "failed_attempts": 0,
        "lockout_until": 0,
        "totp_secret": secrets.token_hex(16),
        "current_otp": None,
        "otp_expiry": 0
    }

    save_user_data(users)
    print(f"User {username} registered successfully.")
    return True


#Verifies user login info and handles lockout scenario
def login_user(username, password):
    users = load_user_data()

    #check if user is in the db
    if username not in users:
        print("User not found.")
        return False
    
    #find user entry in db and get timestamp
    user = users[username]
    now = time.time()

    #check lockout time (if there is one or if expired)
    if now < user['lockout_until']:
        remaining = int((user['lockout_until'] - now) // 60)
        print(f"Account locked. Try again in {remaining} minutes.")
        return False
    
    # argon2 constant time comparison 
    try:
        ph.verify(user['password'], password)
        
        # Reset attempts on success
        user['failed_attempts'] = 0
        save_user_data(users)
        return True
    #if failed attempt increment failure and set lockout
    except VerifyMismatchError:
        user['failed_attempts'] += 1
        print(f"Invalid password. Attempt {user['failed_attempts']}/{5}")
        
        if user['failed_attempts'] >= 5: #five lockout attempts
            user['lockout_until'] = now + 900 #15 minute lockout time
            print("Account locked for 15 minutes.")
        
        #save to file 
        save_user_data(users)
        return False
    #memory cleanup of plaintext pw (just in case)
    finally:
        del password

In [ ]:
# main 

def main ():
    print()

if __name__ == "__main__":
    main()